# Bronze → Silver: Step-by-Step

Shows each of the 4 fixes applied to in-memory Bronze data.
No MinIO or Delta required — pure PySpark transforms.

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
import datetime

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("silver-walkthrough")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark ready:", spark.version)

: 

In [ ]:
# ── Bronze: orders (Problem A city skew + Problem B NULLs) ──────────────────
T = datetime.datetime
orders = spark.createDataFrame([
    ("O001", "C001", "Ho Chi Minh City", None,       None),        # pre-schema-change
    ("O002", "C002", "Ho Chi Minh City", None,       None),
    ("O003", "C003", "Ho Chi Minh City", None,       None),
    ("O004", "C001", "Ho Chi Minh City", "express",  "PROMO10"),   # post-schema-change
    ("O005", "C004", "Hanoi",           "standard", None),
    ("O006", "C005", "Da Nang",         "same_day", "SALE20"),
], ["order_id", "customer_id", "shipping_city", "shipping_method", "coupon_code"])

# ── Bronze: order_items (Problem C duplicates) ───────────────────────────────
order_items = spark.createDataFrame([
    ("OI001", "O001", "P001", 10.0, T(2026, 2, 1, 10,  0)),   # original
    ("OI001", "O001", "P001", 10.0, T(2026, 2, 1, 10,  5)),   # duplicate — later ts
    ("OI002", "O001", "P002", 25.0, T(2026, 2, 1, 10,  0)),
    ("OI003", "O002", "P003", 15.0, T(2026, 2, 15, 9,  0)),
    ("OI004", "O003", "P001", 10.0, T(2026, 3,  1, 8,  0)),
], ["order_item_id", "order_id", "product_id", "unit_price", "created_ts"])

# ── Bronze: products (small dim used for broadcast join demo) ────────────────
products = spark.createDataFrame([
    ("P001", "electronics", 10.0),
    ("P002", "clothing",    25.0),
    ("P003", "food",        15.0),
], ["product_id", "category", "base_price"])

print("orders:", orders.count(), "rows | order_items:", order_items.count(), "rows")

## Fix 1 — AQE SkewJoin (Problem A: 85% Ho Chi Minh City)

Passive fix — just two config flags on the session.  
Spark detects the skewed partition **at runtime** and splits it automatically.

In [ ]:
print("AQE enabled:  ", spark.conf.get("spark.sql.adaptive.enabled", "false"))
print("skewJoin:     ", spark.conf.get("spark.sql.adaptive.skewJoin.enabled", "false"))

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print("\nAfter Fix 1:")
print("AQE enabled:  ", spark.conf.get("spark.sql.adaptive.enabled"))
print("skewJoin:     ", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))

## Fix 2 — Schema Evolution (Problem B: NULL coupon_code & shipping_method)

Orders created before `schema_change_date` have NULLs in these two columns.  
Silver fills them with sentinel values so downstream Gold joins never see NULLs.

In [ ]:
print("=== BRONZE (before) ===")
orders.select("order_id", "shipping_method", "coupon_code").show()

null_before = orders.filter(F.col("coupon_code").isNull()).count()
print(f"NULL coupon_code rows: {null_before}")

In [ ]:
orders_silver = orders.withColumn(
    "coupon_code",
    F.when(F.col("coupon_code").isNull(), "LEGACY").otherwise(F.col("coupon_code"))
).withColumn(
    "shipping_method",
    F.when(F.col("shipping_method").isNull(), "UNKNOWN").otherwise(F.col("shipping_method"))
)

print("=== SILVER (after) ===")
orders_silver.select("order_id", "shipping_method", "coupon_code").show()

null_after = orders_silver.filter(F.col("coupon_code").isNull()).count()
print(f"NULL coupon_code rows: {null_after}  (was {null_before})")

## Fix 3 — Dedup (Problem C: ~2% duplicate order_items)

Duplicate rows share the same `(order_id, product_id, unit_price)` natural key.  
We keep the **earliest** `created_ts` using a window rank.

In [ ]:
print("=== BRONZE (before) ===")
order_items.show()
print(f"Row count: {order_items.count()}  — OI001 appears twice")

In [ ]:
window = Window.partitionBy("order_id", "product_id", "unit_price").orderBy(F.col("created_ts").asc())

order_items_silver = (
    order_items
    .withColumn("_rank", F.row_number().over(window))
    .filter(F.col("_rank") == 1)
    .drop("_rank")
)

print("=== SILVER (after) ===")
order_items_silver.show()
print(f"Row count: {order_items_silver.count()}  (was {order_items.count()}, earliest ts kept)")

## Fix 4 — Broadcast Join (Problem A: cardinality)

`products` is a small dimension (~45k rows, ~5 MB).  
Without a hint Spark uses a SortMergeJoin (expensive shuffle).  
With `broadcast()` Spark sends products to every executor — no shuffle.

In [ ]:
print("=== WITHOUT broadcast hint ===")
no_hint = order_items.join(products, on="product_id", how="left")
no_hint.explain(mode="simple")

In [ ]:
print("=== WITH broadcast hint ===")
with_hint = order_items.join(broadcast(products), on="product_id", how="left")
with_hint.explain(mode="simple")

print("\nResult sample:")
with_hint.select("order_item_id", "product_id", "category", "unit_price").show()

## Summary

| Fix | Problem | What changed |
|-----|---------|---------------|
| 1 — AQE skewJoin | A: 85% HCMC | Session config — Spark splits skewed partitions at runtime |
| 2 — NULL fill | B: schema evolution | `coupon_code` NULL → `LEGACY`, `shipping_method` NULL → `UNKNOWN` |
| 3 — Dedup | C: 2% duplicate rows | Keep earliest `created_ts` per `(order_id, product_id, unit_price)` |
| 4 — Broadcast | A: products join | `SortMergeJoin` (shuffle) → `BroadcastHashJoin` (no shuffle) |